# Chapter 12 - Case Outstanding Development Technique

> In "Loss Reserving," Ronald Wiser describes a development approach that incorporates the
> historical relationships between paid claims and case outstanding. Mr. Wiser states: "The reserve
> development method attempts to analyze the adequacy of case reserves based on the history of
> payments against those case reserves."
>
> -- Friedland, Chapter 12

The **Case Outstanding Development Technique** (Approach #1, Ronald Wiser's method) examines
the historical relationship between claim payments and carried case reserves, as well as the runoff
behavior of case reserves themselves.

### Key Assumptions & Mechanics
1. **IBNR Activity Related to Known Claims**: The technique assumes that future claims emergence
   and development are consistently related to claims already reported (making it particularly well-suited
   for lines where reporting happens early, or for claims-made / report year analyses).
2. **Two Projection Ratios**:
   - **Ratio of Incremental Paid Claims to Previous Case Outstanding**:
     $$\text{Ratio}_{\text{paid}}(w, d) = \frac{\text{Incremental Paid}(w, d)}{\text{Case Outstanding}(w, d-12)}$$
     Measures what proportion of case reserves at age $d-12$ are paid out during the development interval $(d-12 \to d)$.
   - **Ratio of Case Outstanding to Previous Case Outstanding**:
     $$\text{Ratio}_{\text{case}}(w, d) = \frac{\text{Case Outstanding}(w, d)}{\text{Case Outstanding}(w, d-12)}$$
     Measures the runoff/decay rate of case reserves from age $d-12$ to $d$.
3. **Completing the Square**: Selected case runoff ratios complete the case outstanding square,
   which then drives the incremental paid claims projection to ultimate.

In `chainladder-python`, the deterministic framework for this technique is provided by `cl.CaseOutstanding`.
This notebook recreates the exhibits from Friedland Chapter 12:

- **Exhibit I** - U.S. Industry Auto (Sheets 1 & 2)


In [1]:
import numpy as np
import pandas as pd
import chainladder as cl
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


## Exhibit I - U.S. Industry Auto

We begin with **U.S. Industry Auto** (Exhibit I), valued at 12/31/2007 across accident years 1998–2007.
The source reported (incurred) and cumulative paid triangles come from Chapter 7 (`cl.load_sample("usauto")`).


### Exhibit I, Sheet 1: Case Outstanding and Incremental Paid Claims ($000)

In Sheet 1, two fundamental triangles are derived:
1. **Case Outstanding**: Reported (incurred) claims minus cumulative paid claims:
   $$\text{Case Outstanding}_{w, d} = \text{Incurred}_{w, d} - \text{Paid}_{w, d}$$
2. **Incremental Paid Claims**: Period-to-period payments:
   $$\text{Incremental Paid}_{w, d} = \text{Paid}_{w, d} - \text{Paid}_{w, d-12} \quad (d > 12)$$


In [2]:
usauto = cl.load_sample("usauto")
paid_cumulative = usauto["paid"]
incurred = usauto["incurred"]

# 1. Triangle of Case Outstanding ($000)
case_outstanding = incurred - paid_cumulative

# 2. Triangle of Incremental Paid Claims ($000)
incremental_paid = paid_cumulative.cum_to_incr()

# Format as DataFrames for display
case_df = case_outstanding.to_frame(origin_as_datetime=False)
case_df.columns = [int(col) for col in case_df.columns]
case_df.index = [int(getattr(idx, "year", idx)) for idx in case_df.index]

incr_paid_df = incremental_paid.to_frame(origin_as_datetime=False)
incr_paid_df.columns = [int(col) for col in incr_paid_df.columns]
incr_paid_df.index = [int(getattr(idx, "year", idx)) for idx in incr_paid_df.index]

print("Case Outstanding as of (months):")
display(case_df.style.format("{:,.0f}", na_rep=""))

print("Incremental Paid Claims as of (months):")
display(incr_paid_df.style.format("{:,.0f}", na_rep=""))


Case Outstanding as of (months):


,12,24,36,48,60,72,84,96,108,120
1998,"18,478,233","9,937,970","5,506,911","2,892,519","1,440,783","767,842","413,097","242,778","169,222","98,117"
1999,"18,544,291","9,955,034","5,623,522","3,060,431","1,520,760","764,736","443,528","284,732","185,233",
2000,"19,034,933","10,395,464","5,969,194","3,217,937","1,567,806","842,849","457,854","304,704",,
2001,"19,401,810","10,487,914","5,936,461","3,056,202","1,532,147","777,926","421,141",,,
2002,"20,662,461","11,176,330","6,198,509","3,350,967","1,609,188","785,497",,,,
2003,"21,078,651","11,098,119","6,398,219","3,431,210","1,634,690",,,,,
2004,"21,047,539","11,150,459","6,316,995","3,201,985",,,,,,
2005,"21,260,172","11,087,832","6,141,416",,,,,,,
2006,"20,973,908","11,034,842",,,,,,,,
2007,"21,623,594",,,,,,,,,


Incremental Paid Claims as of (months):


,12,24,36,48,60,72,84,96,108,120
1998,"18,539,254","14,691,785","6,830,969","3,830,031","2,004,496","868,887","455,900","225,555","108,579","88,731"
1999,"20,410,193","15,680,491","7,168,718","3,899,839","2,049,291","953,511","463,714","253,051","121,726",
2000,"22,120,843","16,855,171","7,413,268","4,173,103","2,172,895","1,004,821","544,233","248,891",,
2001,"22,992,259","17,103,939","7,671,637","4,326,081","2,269,520","1,015,365","499,620",,,
2002,"24,092,782","17,702,531","8,108,490","4,449,081","2,401,492","1,052,839",,,,
2003,"24,084,451","17,315,161","7,670,720","4,513,869","2,346,453",,,,,
2004,"24,369,770","17,120,093","7,746,815","4,537,994",,,,,,
2005,"25,100,697","17,601,532","7,942,765",,,,,,,
2006,"25,608,776","17,997,721",,,,,,,,
2007,"27,229,969",,,,,,,,,


### Exhibit I, Sheet 2: Ratio of Incremental Paid Claims to Previous Case Outstanding

Sheet 2 computes the proportion of claims paid during each development interval relative to the case outstanding
at the beginning of that interval:
$$\text{Ratio}_{w, d} = \frac{\text{Incremental Paid}_{w, d}}{\text{Case Outstanding}_{w, d-12}} \quad \text{for } d \in \{24, 36, \dots, 120\}$$

Averages evaluated across available historical years include:
- **Simple Average (Latest 5 years)**
- **Simple Average (Latest 3 years)**
- **Medial Average (Latest 5x1)**: Trimming the single highest and lowest values from the latest 5 observations before averaging.

The selected ratio uses the **Latest 3** simple average for all maturities (24 to 120 months) and judgmentally selects **1.100** for **To Ult** (assuming 10% more than the 120-month case outstanding will ultimately be paid).


In [3]:
# Calculate unrounded ratios: Incremental Paid at age d / Case Outstanding at age d-12
case_prior = case_outstanding.iloc[..., :-1]
incr_paid_dev = incremental_paid.iloc[..., 1:]

# Ratio DataFrame (unrounded calculations)
ratio_paid_to_case = (
    incr_paid_dev.to_frame(origin_as_datetime=False)
    / case_prior.to_frame(origin_as_datetime=False).values
)
ratio_paid_to_case.columns = [int(col) for col in incr_paid_dev.ddims]
ratio_paid_to_case.index = [int(getattr(idx, "year", idx)) for idx in ratio_paid_to_case.index]

# Unrounded Averages
def calc_medial_5x1(s):
    vals = s.dropna().tail(5).values
    if len(vals) > 2:
        return np.sort(vals)[1:-1].mean()
    elif len(vals) > 0:
        return vals.mean()
    return np.nan

latest_5_avg = ratio_paid_to_case.apply(lambda s: s.dropna().tail(5).mean())
latest_3_avg = ratio_paid_to_case.apply(lambda s: s.dropna().tail(3).mean())
medial_5x1_avg = ratio_paid_to_case.apply(calc_medial_5x1)

# Summary table of averages
averages_df = pd.DataFrame({
    "Latest 5": latest_5_avg,
    "Latest 3": latest_3_avg,
    "Latest 5x1": medial_5x1_avg,
}).T
averages_df["To Ult"] = np.nan

# Selected Ratios
selected_ratios = latest_3_avg.copy()
selected_ratios["To Ult"] = 1.100
selected_df = pd.DataFrame([selected_ratios], index=["Selected"])

print("Ratio of Incremental Paid Claims to Previous Case Outstanding:")
display(ratio_paid_to_case.style.format("{:.3f}", na_rep=""))

print("Averages of the Ratio of Incremental Paid Claims to Previous Case Outstanding:")
display(averages_df.style.format("{:.3f}", na_rep=""))

print("Selected Ratio of Incremental Paid Claims to Previous Case Outstanding:")
display(selected_df.style.format("{:.3f}", na_rep=""))


Ratio of Incremental Paid Claims to Previous Case Outstanding:


,24,36,48,60,72,84,96,108,120
1998,0.795,0.687,0.695,0.693,0.603,0.594,0.546,0.447,0.524
1999,0.846,0.720,0.693,0.670,0.627,0.606,0.571,0.428,
2000,0.885,0.713,0.699,0.675,0.641,0.646,0.544,,
2001,0.882,0.731,0.729,0.743,0.663,0.642,,,
2002,0.857,0.726,0.718,0.717,0.654,,,,
2003,0.821,0.691,0.705,0.684,,,,,
2004,0.813,0.695,0.718,,,,,,
2005,0.828,0.716,,,,,,,
2006,0.858,,,,,,,,
2007,,,,,,,,,


Averages of the Ratio of Incremental Paid Claims to Previous Case Outstanding:


,24,36,48,60,72,84,96,108,120,To Ult
Latest 5,0.836,0.712,0.714,0.698,0.638,0.622,0.553,0.437,0.524,
Latest 3,0.833,0.701,0.714,0.714,0.653,0.631,0.553,0.437,0.524,
Latest 5x1,0.835,0.712,0.714,0.692,0.641,0.624,0.546,0.437,0.524,


Selected Ratio of Incremental Paid Claims to Previous Case Outstanding:


,24,36,48,60,72,84,96,108,120,To Ult
Selected,0.833,0.701,0.714,0.714,0.653,0.631,0.553,0.437,0.524,1.100


### Chainladder Python Integration & Observations

In `chainladder`, `cl.CaseOutstanding` can fit the historical triangle directly and produce `paid_ldf_`:


In [4]:
# Fit cl.CaseOutstanding with Latest 3 average
model = cl.CaseOutstanding(
    paid_to_incurred=("paid", "incurred"),
    paid_n_periods=3,
    case_n_periods=3,
).fit(usauto)

# Display model.paid_ldf_
cl_paid_ldf = model.paid_ldf_.to_frame(origin_as_datetime=False)
print("chainladder CaseOutstanding.paid_ldf_ (Latest 3 average):")
display(cl_paid_ldf.round(3))

# Compare differences with unrounded calculation
diff = np.abs(model.paid_ldf_.values.flatten() - latest_3_avg.values)
print(f"Maximum absolute difference against unrounded calculation: {diff.max():.2e}")


chainladder CaseOutstanding.paid_ldf_ (Latest 3 average):


,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
(All),0.833,0.701,0.714,0.714,0.653,0.631,0.553,0.437,0.524


Maximum absolute difference against unrounded calculation: 0.00e+00


#### Technical Notes & Potential Upstream Issues

1. **Weighting Mask on Ratio Inspection**:
   In `CaseOutstanding`, `model.paid_to_prior_case_` and `model.case_to_prior_case_` apply the internal weighting array `w_`. When `paid_n_periods=3` is passed, older accident years are set to zero in the property output instead of displaying the historical ratios. To inspect the full historical triangle, one must initialize with `paid_n_periods=-1` or compute it on the unmasked triangle.

2. **Column Development Interval Labeling**:
   Because `paid_tri.cum_to_incr().iloc[..., 1:]` has `ddims = [24, 36, 48, ...]`, setting `is_pattern = True` causes the string formatter to render the development column headers as `24-36`, `36-48`, ..., `120-132` rather than `12-24`, `24-36`, ..., `108-120`.

3. **Medial Averages and Tail Selections**:
   `CaseOutstanding` currently implements simple averages over `n_periods`. Support for medial averages (e.g. `5x1`) or explicit tail factors ("To Ult" ratios such as 1.100 for paid and 0.000 for case) is not natively configurable within `CaseOutstanding`.
